# Recommendation System Using Cosine Similarity (Anime Dataset)

## Introduction:

- Data Description:

Unique ID of each anime.
Anime title.
Anime broadcast type, such as TV, OVA, etc.
anime genre.
The number of episodes of each anime.
The average rating for each anime compared to the number of users who gave ratings.


Number of community members for each anime.
Objective:
The objective of this assignment is to implement a recommendation system using cosine similarity on an anime dataset. 
Dataset:
Use the Anime Dataset which contains information about various anime, including their titles, genres,No.of episodes and user ratings etc.

- Tasks:

1. Data Preprocessing:

Load the dataset into a suitable data structure (e.g., pandas DataFrame).
Handle missing values, if any.
Explore the dataset to understand its structure and attributes.

2. Feature Extraction:

Decide on the features that will be used for computing similarity (e.g., genres, user ratings).
Convert categorical features into numerical representations if necessary.
Normalize numerical features if required.

3. Recommendation System:

Design a function to recommend anime based on cosine similarity.
Given a target anime, recommend a list of similar anime based on cosine similarity scores.
Experiment with different threshold values for similarity scores to adjust the recommendation list size.
Analyze the performance of the recommendation system and identify areas of improvement.

4. Interview Questions:
   1. Can you explain the difference between user-based and item-based collaborative filtering?
   2. What is collaborative filtering, and how does it work?

## Importing Libraries and Dataset

In [2]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

In [3]:
anime = pd.read_csv("anime.csv")
anime.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [4]:
anime.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [5]:
anime.describe()

,anime_id,rating,members
count,12294.000000,12064.000000,1.229400e+04
mean,14058.221653,6.473902,1.807134e+04
std,11455.294701,1.026746,5.482068e+04
min,1.000000,1.670000,5.000000e+00
25%,3484.250000,5.880000,2.250000e+02
50%,10260.500000,6.570000,1.550000e+03
75%,24794.500000,7.180000,9.437000e+03
max,34527.000000,10.000000,1.013917e+06


In [6]:
anime.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

## Data Preprocessing

In [7]:
anime = anime[['anime_id', 'name', 'genre', 'rating', 'members']]
anime.dropna(inplace=True)

anime['genre'] = anime['genre'].str.lower()
anime.dropna(inplace=True)

anime.reset_index(drop=True, inplace=True)

## Train-Test Split

In [8]:
train_data, test_data = train_test_split(
    anime, test_size=0.2, random_state=42
)

train_data.reset_index(drop=True, inplace=True)
test_data.reset_index(drop=True, inplace=True)

## Feature Extraction

In [9]:
# Genre Vectorization (TF-IDF)

tfidf = TfidfVectorizer(stop_words='english')
genre_matrix = tfidf.fit_transform(train_data['genre'])

In [10]:
# Normalize Numerical Features

scaler = MinMaxScaler()

numerical_features = scaler.fit_transform(
    train_data[['rating', 'members']]
)

In [11]:
# Combine Features

from scipy.sparse import hstack
feature_matrix = hstack([genre_matrix, numerical_features])

## Compute Cosine Similarity

In [12]:
cosine_sim = cosine_similarity(feature_matrix, feature_matrix)

In [13]:
# Create Index Mapping
# This helps quickly locate anime indices by name.

indices = pd.Series(train_data.index, index=train_data['name']).drop_duplicates()

In [14]:
# Recommendation Function (with Threshold)

def recommend(anime_name, top_n=10, threshold=0.3):
    if anime_name not in indices:
        return []

    idx = indices[anime_name]
    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = [s for s in sim_scores if s[1] >= threshold]
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:top_n+1]
    anime_indices = [i[0] for i in sim_scores]

    return train_data['name'].iloc[anime_indices].tolist()

In [15]:
# Experiment with Different Thresholds
# Threshold controls recommendation quality and quantity.

thresholds = [0.2, 0.3, 0.4, 0.5]

for t in thresholds:
    print(f"\nThreshold: {t}")
    print(recommend("Naruto", threshold=t))


Threshold: 0.2
['Naruto: Shippuuden', 'Dragon Ball Z', 'Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsugu Mono', 'Boruto: Naruto the Movie', 'Naruto x UT', 'Boruto: Naruto the Movie - Naruto ga Hokage ni Natta Hi', 'Naruto Shippuuden: Sunny Side Battle', 'Dragon Ball Kai', 'Dragon Ball Super', 'Naruto: Shippuuden Movie 6 - Road to Ninja']

Threshold: 0.3
['Naruto: Shippuuden', 'Dragon Ball Z', 'Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsugu Mono', 'Boruto: Naruto the Movie', 'Naruto x UT', 'Boruto: Naruto the Movie - Naruto ga Hokage ni Natta Hi', 'Naruto Shippuuden: Sunny Side Battle', 'Dragon Ball Kai', 'Dragon Ball Super', 'Naruto: Shippuuden Movie 6 - Road to Ninja']

Threshold: 0.4
['Naruto: Shippuuden', 'Dragon Ball Z', 'Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsugu Mono', 'Boruto: Naruto the Movie', 'Naruto x UT', 'Boruto: Naruto the Movie - Naruto ga Hokage ni Natta Hi', 'Naruto Shippuuden: Sunny Side Battle', 'Dragon Ball Kai', 'Dragon Ball Super', 'Naruto: Shippuuden Mov

## Model Evaluation

In [16]:
# Index Mapping
indices = pd.Series(train_data.index, index=train_data['name']).drop_duplicates()

In [17]:
# Instead of predicting classes directly, cosine similarity is treated like probability scores.
query_anime = "Naruto"
query_idx = indices[query_anime]

similarity_scores = cosine_sim[query_idx]

In [18]:
# Relevant anime = same main genre as query anime
query_genre = train_data.loc[
    train_data['name'] == query_anime, 'genre'
].values[0]

main_genre = query_genre.split(',')[0].strip()

y_test = test_data['genre'].str.contains(
    main_genre, case=False, na=False
).astype(int).values

In [19]:
# Convert Similarity → Binary Prediction (KEY STEP)

threshold = 0.6
# Align similarity scores with test set
test_indices = test_data.index.intersection(train_data.index)
y_prob = similarity_scores[test_indices]
y_pred = (y_prob >= threshold).astype(int)

In [21]:
# Evaluation Metrics
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
precision = precision_score(y_test[:len(y_pred)], y_pred, zero_division=0)
recall = recall_score(y_test[:len(y_pred)], y_pred, zero_division=0)
f1 = f1_score(y_test[:len(y_pred)], y_pred, zero_division=0)

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

print("\nClassification Report:\n")
print(classification_report(y_test[:len(y_pred)], y_pred))

Precision: 0.2391304347826087
Recall: 0.039568345323741004
F1-score: 0.06790123456790123

Classification Report:

              precision    recall  f1-score   support

           0       0.77      0.96      0.85      1848
           1       0.24      0.04      0.07       556

    accuracy                           0.75      2404
   macro avg       0.50      0.50      0.46      2404
weighted avg       0.65      0.75      0.67      2404



### 📊 Interpretation of Classification Report (Threshold = 0.6)

This classification report evaluates the recommendation system when a similarity threshold of **0.6** is applied. At this threshold, the system becomes more selective, recommending only anime with high similarity scores.

---

#### 🔢 Overall Performance Metrics

- **Precision (Class 1 – Recommended): 0.24**  
  Only **24% of the anime recommended by the system are actually relevant**.  
  This indicates that while recommendations are more selective, many are still not relevant.

- **Recall (Class 1 – Recommended): 0.04**  
  The system successfully identifies **only 4% of all relevant anime**.  
  This shows that the high threshold causes the system to **miss most relevant items**.

- **F1-Score (Class 1): 0.07**  
  The low F1-score reflects a **poor balance between precision and recall**, mainly due to very low recall.

---

#### 🧩 Class-wise Interpretation

#### 🔹 Class 0 (Not Recommended)
- **Precision: 0.77**  
  When the system predicts an anime as *not recommended*, it is correct **77% of the time**.
- **Recall: 0.96**  
  The system correctly identifies **96% of non-relevant anime**.
- **F1-Score: 0.85**  
  Strong performance for filtering out non-relevant items.

✔ This shows the system is **very effective at rejecting irrelevant anime**.

---

#### 🔹 Class 1 (Recommended)
- **Precision: 0.24**  
  Out of all anime marked as recommended, only **24% are truly relevant**.
- **Recall: 0.04**  
  The system captures **very few relevant anime**.
- **F1-Score: 0.07**  
  Indicates **weak recommendation performance** at this threshold.

❌ This means the system is **too strict**, resulting in very few correct recommendations.

---

#### ⚖️ Accuracy and Averages

- **Accuracy: 0.75**  
  The model appears accurate, but this is misleading due to **class imbalance** (many more non-relevant anime).

- **Macro Average**  
  - Precision: 0.50  
  - Recall: 0.50  
  Treats both classes equally and reveals **imbalanced performance**.

- **Weighted Average**  
  - Precision: 0.65  
  - Recall: 0.75  
  Influenced by the dominant class (Class 0), hence overestimates performance.

---

#### ⚠️ Key Insight

Although the accuracy is relatively high, the **recommendation quality is poor**.  
The system favors **not recommending anime**, which inflates accuracy but reduces usefulness.

---

#### ✅ Conclusion

At a threshold of **0.6**, the recommendation system prioritizes precision and filters out most items. However, this results in extremely low recall and poor F1-score for relevant anime. Therefore, this threshold is **not suitable for effective recommendations**. Lower thresholds (e.g., 0.4) offer a better balance between precision and recall.

---


In [22]:
for threshold in [0.4, 0.5, 0.6, 0.7]:
    y_pred = (y_prob >= threshold).astype(int)
    
    print(f"\nThreshold = {threshold}")
    print("Precision:", precision_score(y_test[:len(y_pred)], y_pred, zero_division=0))
    print("Recall:", recall_score(y_test[:len(y_pred)], y_pred, zero_division=0))
    print("F1:", f1_score(y_test[:len(y_pred)], y_pred, zero_division=0))


Threshold = 0.4
Precision: 0.1955193482688391
Recall: 0.17266187050359713
F1: 0.1833810888252149

Threshold = 0.5
Precision: 0.21019108280254778
Recall: 0.05935251798561151
F1: 0.09256661991584852

Threshold = 0.6
Precision: 0.2391304347826087
Recall: 0.039568345323741004
F1: 0.06790123456790123

Threshold = 0.7
Precision: 0.2692307692307692
Recall: 0.012589928057553957
F1: 0.024054982817869417


### 📊 Interpretation of Threshold-Based Evaluation Results

The table below summarizes the performance of the recommendation system at different similarity thresholds:

| Threshold | Precision | Recall | F1-Score |
|----------|-----------|--------|----------|
| 0.4 | 0.196 | 0.173 | 0.183 |
| 0.5 | 0.210 | 0.059 | 0.093 |
| 0.6 | 0.239 | 0.040 | 0.068 |
| 0.7 | 0.269 | 0.013 | 0.024 |

---

#### 🔍 Key Observations

- As the **similarity threshold increases**, **precision consistently improves**.
- At the same time, **recall sharply decreases** with higher thresholds.
- The **F1-score declines** as the threshold increases, indicating a growing imbalance between precision and recall.

---

#### 📌 Threshold-wise Interpretation

- **Threshold = 0.4**  
  - Highest recall and F1-score among all thresholds.
  - Recommends more anime, capturing a larger portion of relevant items.
  - Suitable when **coverage** is more important than accuracy.

- **Threshold = 0.5**  
  - Moderate precision but significant drop in recall.
  - System becomes more selective, missing many relevant anime.

- **Threshold = 0.6**  
  - Higher precision indicates more accurate recommendations.
  - Very low recall suggests that only a small subset of relevant anime is recommended.
  - Suitable when **quality is preferred over quantity**.

- **Threshold = 0.7**  
  - Highest precision but extremely low recall.
  - Very few recommendations are made.
  - Not ideal due to poor overall balance.

---

#### ⚖️ Trade-off Analysis

- **Lower thresholds** → Higher recall, lower precision  
- **Higher thresholds** → Higher precision, lower recall  

This highlights the classic **precision–recall trade-off** in recommendation systems.

---

#### ✅ Best Threshold Choice

- **Threshold = 0.4** provides the **best balance**, as it achieves the highest F1-score.
- It is the most suitable threshold when both recommendation relevance and coverage are important.

---

### 🚀 Conclusion

The evaluation demonstrates that similarity threshold selection has a significant impact on recommendation performance. A lower threshold allows broader recommendations, while a higher threshold increases confidence at the cost of coverage. Based on the observed metrics, a threshold of **0.4** is the optimal choice for this system.

---
